# 02. Prompt Engineering 실습
**SK하이닉스 Autonomous R&D — Day 3** 

---
## 0. 준비

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

#질문 함수 생성
def ask(system_prompt, user_prompt, temperature=0.2):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content

In [2]:
# 기본적으로 같은 모델이라도 프롬프트에 따라 다른 출력을 생성

system = 'You are a helpful assistant. Answer in Korean.'

prompts_ko = [
    "프랑스의 수도는",
    "Q: 프랑스의 수도는 어디인가요? A:",
    "프랑스의 수도 도시는",
]

for prompt in prompts_ko:
    print(f"프롬프트: {prompt}")
    print(f"생성: {ask(system, prompt, temperature=0.2)}")
    print()

프롬프트: 프랑스의 수도는
생성: 프랑스의 수도는 파리입니다.

프롬프트: Q: 프랑스의 수도는 어디인가요? A:
생성: 프랑스의 수도는 파리입니다.

프롬프트: 프랑스의 수도 도시는
생성: 프랑스의 수도 도시는 파리입니다.



In [ ]:
# 기본적으로 같은 모델이라도 프롬프트에 따라 다른 출력을 생성

system = 'You are a helpful assistant. Answer in Korean.'

prompts_ko1 = [
    "",
    "Q: 프랑스의 수도는 어디인가요? A:",
    "프랑스의 수도 도시는",
]

for prompt in prompts_ko1:
    print(f"프롬프트: {prompt}")
    print(f"생성: {ask(system, prompt, temperature=0.2)}")
    print()

---
## 1. Zero-shot vs Few-shot 

| 방식 | 설명 |
|------|------|
| **Zero-shot** | 예시 없이 지시만 |
| **Few-shot** | 원하는 형식의 **예시**를 함께 제공 |

In [3]:
system = 'You are a helpful assistant. Answer in Korean.'
# Zero-shot — 형식 지시 없음
user_zero = '''
아래 영화 리뷰가 긍정인지 부정인지 판정해줘.
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Zero-shot ===')
print(ask(system, user_zero))

=== Zero-shot ===
첫 번째 리뷰는 부정적인 요소가 포함되어 있어 부정으로 판정할 수 있습니다. 두 번째 리뷰는 긍정적인 내용이므로 긍정으로 판정할 수 있습니다.


In [4]:
# Few-shot — 출력 형식 예시 포함
user_few = '''
아래 형식으로 감성 판정해줘.

[예시]
입력: "배우 연기가 훌륭했다" → 긍정 | 9/10
입력: "스토리가 지루했다" → 부정 | 3/10

[실제 데이터]
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Few-shot ===')
print(ask(system, user_few))

=== Few-shot ===
입력: "연출은 좋았지만 2시간이 너무 길었다" → 긍정 | 6/10  
입력: "최고의 영화, 또 보고 싶다" → 긍정 | 10/10


In [5]:
# Few-shot — 출력 형식 예시 포함
system1 = 'You are a child'
user_few = '''
아래 형식으로 감성 판정해줘.

[예시]
입력: "배우 연기가 훌륭했다" → 긍정 | 9/10
입력: "스토리가 지루했다" → 부정 | 3/10

[실제 데이터]
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Few-shot ===')
print(ask(system1, user_few))

=== Few-shot ===
입력: "연출은 좋았지만 2시간이 너무 길었다" → 중립 | 5/10  
입력: "최고의 영화, 또 보고 싶다" → 긍정 | 10/10


In [6]:
zero_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

문제: 행복한 : 슬픈 :: 좋은 :"""

print(ask(system, zero_shot_prompt_ko, temperature=0.7))

좋은 : 나쁜


In [7]:
few_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :"""

print('Few-shot 프롬프트:')
print(few_shot_prompt_ko)
print('\n생성된 완성:')
print(ask(system, few_shot_prompt_ko, temperature=0.7))

Few-shot 프롬프트:
다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :

생성된 완성:
나쁜


---
## 2. 역할(Role) 설정 

In [8]:
question = '하루 커피 4잔 마셔도 괜찮을까?'

print('=== 일반 assistant ===')
print(ask('You are a helpful assistant.', question))
print('\n' + '=' * 50 + '\n')

=== 일반 assistant ===
하루에 커피 4잔을 마시는 것은 일반적으로 대부분의 성인에게 괜찮다고 여겨집니다. 여러 연구에 따르면, 하루 400mg 이하의 카페인은 건강한 성인에게 안전하다고 알려져 있습니다. 이는 대략 커피 4잔에 해당합니다. 그러나 개인의 카페인 민감도, 건강 상태, 임신 여부 등에 따라 다를 수 있으므로, 자신의 몸 상태를 잘 살펴보는 것이 중요합니다.

카페인을 과다 섭취하면 불안, 불면증, 심박수 증가 등의 부작용이 나타날 수 있으니, 이러한 증상이 나타나면 섭취량을 줄이는 것이 좋습니다. 만약 특정 건강 문제가 있거나 카페인에 민감한 경우, 의사와 상담하는 것이 좋습니다.




In [9]:
print('=== 영양사 역할 ===')
print(ask(
    'You are a registered dietitian. Answer in Korean with caffeine mg estimate and health advice.',
    question,
))

=== 영양사 역할 ===
하루에 커피 4잔을 마시는 것은 일반적으로 건강한 성인에게는 괜찮다고 여겨집니다. 일반적으로 1잔의 커피에는 약 95mg의 카페인이 포함되어 있습니다. 따라서 하루 4잔을 마신다면 약 380mg의 카페인을 섭취하게 됩니다.

미국 식품의약국(FDA)에서는 건강한 성인이 하루에 최대 400mg의 카페인을 섭취하는 것이 안전하다고 권장하고 있습니다. 그러나 개인의 카페인 민감도는 다를 수 있으므로, 카페인에 민감한 사람이나 특정 건강 문제가 있는 경우에는 주의가 필요합니다.

건강을 유지하기 위해 다음과 같은 조언을 드립니다:

1. **수분 섭취**: 커피를 마시는 것 외에도 충분한 물을 섭취하여 수분 균형을 유지하세요.
2. **카페인 섭취 조절**: 카페인에 민감한 경우, 섭취량을 조절하거나 카페인 없는 음료를 선택하는 것도 좋습니다.
3. **수면 관리**: 카페인은 수면에 영향을 줄 수 있으므로, 저녁 시간에는 카페인 섭취를 피하는 것이 좋습니다.
4. **균형 잡힌 식사**: 커피 외에도 다양한 영양소를 포함한 균형 잡힌 식사를 하세요.

개인의 건강 상태에 따라 다를 수 있으므로, 필요하다면 전문가와 상담하는 것이 좋습니다.


In [10]:
roles = [
    "당신은 친절한 고객 서비스 상담원입니다.",
    "당신은 전문적인 기술 문서 작성자입니다.",
    "당신은 창의적인 소설가입니다."
]

question = "인공지능에 대해 설명해주세요."

for role in roles:
    user_prompt = f"{role}\n\n질문: {question}\n\n답변:"
    print(f"=== {role} ===")
    print(f"답변: {ask('Answer in Korean.', user_prompt, temperature=0.8)}")
    print()

=== 당신은 친절한 고객 서비스 상담원입니다. ===
답변: 인공지능(AI)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습하고, 문제를 해결하며, 의사 결정을 할 수 있도록 하는 기술입니다. AI는 다양한 알고리즘과 데이터 분석을 통해 패턴을 인식하고, 경험을 기반으로 스스로 개선할 수 있습니다. 

인공지능은 여러 분야에서 활용되고 있습니다. 예를 들어, 자연어 처리(NLP)를 통해 인간의 언어를 이해하고 소통하는 데 사용되며, 이미지 인식, 자율주행차, 추천 시스템 등 다양한 응용 프로그램에서도 중요한 역할을 하고 있습니다. 

AI는 크게 두 가지 유형으로 나눌 수 있습니다. 하나는 약한 인공지능(Weak AI)으로, 특정 작업에 특화되어 있고, 다른 하나는 강한 인공지능(Strong AI)으로, 인간처럼 다양한 작업을 수행할 수 있는 능력을 갖춘 것입니다. 현재 우리가 사용하는 대부분의 AI는 약한 인공지능에 해당합니다.

인공지능의 발전은 많은 기회를 제공하지만, 동시에 윤리적 문제나 일자리 대체와 같은 도전 과제를 동반하기도 합니다. 이러한 점들을 고려하며 AI 기술을 발전시켜 나가는 것이 중요합니다.

=== 당신은 전문적인 기술 문서 작성자입니다. ===
답변: 인공지능(AI, Artificial Intelligence)은 기계나 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 추론, 문제 해결, 이해 및 자연어 처리와 같은 작업을 수행할 수 있도록 하는 기술을 말합니다. AI는 주로 기계 학습, 딥러닝, 자연어 처리(NLP), 컴퓨터 비전 등 다양한 분야로 나뉘어 발전하고 있습니다.

1. **기계 학습(Machine Learning)**: AI의 하위 분야로, 컴퓨터가 데이터를 통해 학습하고 개선할 수 있도록 하는 알고리즘을 개발하는 것입니다. 기계 학습은 감독 학습, 비감독 학습, 강화 학습 등으로 나눌 수 있습니다.

2. **딥러닝(Deep Learning)**: 기계 학습의 한 종류로, 인공 신경망을 사용하여 데이터에서 패턴을 인식하고

---
## 3. 출력 형식 지정 

In [11]:
topic = '재택근무의 장단점 3가지'
system_ko = 'Answer in Korean.'

print('=== Markdown ===')
print(ask(system_ko, topic + '\n형식: markdown bullet point'))
print('\n' + '=' * 50 + '\n')

=== Markdown ===
### 재택근무의 장단점

#### 장점
- **유연한 근무 시간**: 개인의 일정에 맞춰 근무 시간을 조정할 수 있어 일과 삶의 균형을 맞추기 용이함.
- **교통비 및 시간 절약**: 출퇴근 시간을 줄이고 교통비를 절감할 수 있어 경제적이고 효율적임.
- **편안한 근무 환경**: 자신에게 맞는 환경에서 일할 수 있어 집중력과 생산성이 향상될 수 있음.

#### 단점
- **사회적 고립감**: 동료와의 직접적인 소통이 줄어들어 외로움을 느낄 수 있음.
- **업무와 개인 생활의 경계 모호**: 집에서 일하다 보면 업무와 개인 생활의 경계가 흐려져 스트레스를 받을 수 있음.
- **자기 관리의 어려움**: 자율성이 높아지면서 업무를 관리하는 데 어려움을 겪을 수 있음.




In [12]:
print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (항목 | 장점 | 단점)만 출력',
))

=== Table ===
| 항목       | 장점                       | 단점                       |
|------------|----------------------------|----------------------------|
| 유연성     | 근무 시간을 자유롭게 조정 가능 | 일과 개인 생활의 경계 모호 |
| 교통비 절감 | 출퇴근 시간 및 비용 절감    | 사회적 고립감 증가         |
| 생산성     | 집중할 수 있는 환경 조성    | 자율성 부족 시 생산성 저하  |


---
### 출력 형식 (표 / JSON)

In [13]:
import json

topic = '식각(Etch) 공정에서 수율에 영향 주는 요인 3가지'
system_ko = 'Answer in Korean.'

print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (요인 | 설명 | 대응)만 출력',
))

=== Table ===
| 요인         | 설명                                   | 대응                          |
|--------------|--------------------------------------|-------------------------------|
| 화학물질     | 식각 공정에 사용되는 화학물질의 농도와 조성이 수율에 영향을 미침 | 화학물질의 품질 관리 및 최적화 |
| 온도         | 공정 중 온도가 적절하지 않으면 식각 속도와 균일성이 저하됨 | 온도 모니터링 및 제어 시스템 구축 |
| 마스크 품질 | 마스크의 결함이나 불균일성이 식각 결과에 영향을 미침 | 마스크 제작 공정 개선 및 검사 강화 |


In [14]:
print('=== JSON ===')
result = ask(
    'Output ONLY valid JSON. No markdown.',
    topic + '\n{"factors": [{"name": "", "action": ""}]} 형식으로',
)
print(result)
try:
    print('\n파싱 성공:', list(json.loads(result).keys()))
except json.JSONDecodeError:
    print('\n파싱 실패 — ONLY valid JSON 강조 필요')

=== JSON ===
{
  "factors": [
    {
      "name": "화학물질 농도",
      "action": "적절한 농도를 유지하여 식각 속도와 균일성을 확보"
    },
    {
      "name": "식각 시간",
      "action": "최적의 식각 시간을 설정하여 과식각 또는 부족식각 방지"
    },
    {
      "name": "온도",
      "action": "온도를 조절하여 반응 속도와 품질을 향상"
    }
  ]
}

파싱 성공: ['factors']


---
## 4. Chain-of-Thought (CoT)

CoT는 모델이 단계별로 추론 과정을 보여주도록 하는 기법.

In [15]:
## zero-shot 예시

problem = '''
A팀 10명, B팀 15명, C팀 5명.
전체 25명 중 40% 이상이 A팀이면 A팀에 추가 인원 배치.
지금 추가 배치가 필요한가?
'''
system = 'You are a helpful assistant. Answer in Korean.'
print('=== CoT 없음 ===')
print(ask(system, problem))

=== CoT 없음 ===
전체 25명 중 40%는 10명입니다. 현재 A팀은 10명으로, 전체 인원의 40%에 해당합니다. 따라서 A팀에 추가 배치가 필요하지 않습니다. A팀의 인원은 40% 이상이 아니기 때문입니다.


In [16]:
print('=== CoT 적용 ===')
print(ask(
    system + ' Show step-by-step reasoning before final answer.',
    problem + '\n\n단계별로 생각해 봅시다.',
))

=== CoT 적용 ===
1. **전체 인원 확인**: A팀은 10명, B팀은 15명, C팀은 5명으로, 전체 인원은 10 + 15 + 5 = 30명입니다.

2. **A팀 비율 계산**: A팀의 인원 수는 10명입니다. 전체 인원 30명 중 A팀의 비율을 계산합니다.
   \[
   \text{A팀 비율} = \frac{\text{A팀 인원}}{\text{전체 인원}} \times 100 = \frac{10}{30} \times 100 = 33.33\%
   \]

3. **40% 기준 확인**: 문제에서 언급한 바와 같이, A팀의 비율이 40% 이상이어야 추가 인원 배치가 필요합니다. 현재 A팀의 비율은 33.33%로 40%에 미치지 않습니다.

4. **결론 도출**: A팀의 비율이 40% 이상이 아니므로, 추가 배치가 필요하지 않습니다.

따라서, **A팀에 추가 배치가 필요하지 않습니다.**


In [19]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

답: 3,600원


In [17]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

천천히 단계별로 생각해봅시다.

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

문제를 단계별로 해결해보겠습니다.

1. 연필의 가격 계산:
   - 연필 1자루의 가격은 500원입니다.
   - 연필 5자루의 가격은 500원 × 5자루 = 2500원입니다.

2. 지우개의 가격 계산:
   - 지우개 1개의 가격은 300원입니다.
   - 지우개 3개의 가격은 300원 × 3개 = 900원입니다.

3. 총 금액 계산:
   - 연필과 지우개의 총 금액은 2500원 + 900원 = 3400원입니다.

따라서, 총 금액은 3400원입니다.

답: 3400원


In [20]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

답: 6000원

---

문제: 한 상점에서 귤 1개와 바나나 12개를 샀습니다. 사과는 개당 1000원, 배는 개당 200원입니다. 총 금액은 얼마인가요?

답: 3400원

---


문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: """

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

답: 3900원


In [21]:
## one-shot 예시

problem = """다음 문제를 단계별로 풀어보세요.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

단계별 풀이:
1. 사과 3개의 가격: 3 × 1000 = 3000원
2. 배 2개의 가격: 2 × 1500 = 3000원
3. 총 금액: 3000 + 3000 = 6000원
답: 6000원

---

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

단계별 풀이:"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

단계별 풀이:

1. 연필 5자루의 가격: 5 × 500 = 2500원
2. 지우개 3개의 가격: 3 × 300 = 900원
3. 총 금액: 2500 + 900 = 3400원

답: 3400원


---
## 5. System + User 프롬프트 

In [22]:
system_prompt = '''
너는 스타트업 PM의 일정 관리 AI다.
규칙: 결론 먼저, bullet 3개, 한국어
'''

user_prompt = '''
이번 주 마감: 월-기획안, 수-코드리뷰, 금-발표.
목요일에 하루 종일 회의가 잡혔어. 우선순위 조정해줘.
'''

print(ask(system_prompt, user_prompt))

결론: 목요일 회의에 맞춰 일정 조정 필요.

- 월요일: 기획안 마감은 그대로 유지
- 수요일: 코드리뷰를 목요일로 연기
- 금요일: 발표는 그대로 유지, 목요일 회의 후 준비 가능


In [23]:
system_md = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤 "
    "리스크와 가정을 명시해야 한다."
)

user_md = "이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘."

answer = ask(system_md, user_md, temperature=0.7)
print(answer)

| 결론                                   | 리스크                                   | 가정                                   |
|--------------------------------------|---------------------------------------|--------------------------------------|
| VIP 이탈 상위 200명에 대해 맞춤형 재참여 캠페인을 실시하여 이탈 방지 및 고객 충성도를 높인다. | 캠페인 반응이 예상보다 저조할 경우, ROI 감소가 발생할 수 있다. | 고객 이탈의 주요 원인이 특정 제품이나 서비스의 품질 저하라고 가정한다. |
| 1. 고객 데이터 분석: 이탈 이유 및 구매 패턴 파악 | 고객 데이터의 불완전성으로 인한 분석 오류 가능성 | 이탈 고객의 데이터가 충분히 수집되어 있다고 가정한다. |
| 2. 맞춤형 제안: 이탈 고객에게 맞춤형 혜택 제공 | 고객의 선호도가 변화하여 맞춤형 제안이 효과가 없을 수 있음 | 고객의 이전 구매 이력을 바탕으로 혜택이 효과적일 것이라고 가정한다. |
| 3. 재참여 유도: 이메일 및 SMS 캠페인 진행 | 스팸으로 인식되어 고객의 반감을 살 우려 | 고객이 이메일 및 SMS를 수신하는 데 긍정적일 것이라고 가정한다. |
| 4. 피드백 수집: 캠페인 후 고객 피드백 요청 | 고객이 피드백에 응답하지 않을 위험 | 고객이 자신의 의견을 공유할 의향이 있다고 가정한다. |
| 5. 성과 분석: 캠페인 후 이탈률 변화 분석 | 성과 분석이 부정확할 경우, 잘못된 결론 도출 | 이탈률 변화가 캠페인 효과를 제대로 반영한다고 가정한다. |


---
## 6. Self-check Prompting

In [24]:
check_prompt = f"""아래는 네가 작성한 'VIP 이탈 상위 200명 액션 플랜' 초안이다.

[초안]
{answer}

이제 아래 체크리스트로 점검하고, 필요하면 수정본을 작성하라.

체크리스트:
1) 논리적 모순/비약이 있는가?
2) 실행 불가능하거나 모호한 액션이 있는가? (담당/기한/방법이 불명확한지)
3) 과도한 가정이 있는가?
4) 표 형식이 지켜졌는가? (결론 먼저, 리스크/가정 포함)

출력 규칙:
- 먼저 "점검 결과"를 bullet로 간단히 쓰고
- 그 다음 "수정본"을 작성하라
- 수정본은 반드시 표 형태 + 결론 먼저 + 리스크/가정 명시
"""

final_answer = ask(system_md, check_prompt, temperature=0.7)
print('\n--- Self-check 후 최종 답변(수정본) ---')
print(final_answer)


--- Self-check 후 최종 답변(수정본) ---
### 점검 결과
- 논리적 모순이나 비약은 없음.
- 실행 불가능하거나 모호한 액션은 없음.
- 가정이 다소 과도하게 설정된 부분이 있음.
- 표 형식이 지켜졌음.

### 수정본
| 결론                                   | 리스크                                   | 가정                                   |
|--------------------------------------|---------------------------------------|--------------------------------------|
| VIP 이탈 상위 200명에 대해 맞춤형 재참여 캠페인을 실시하여 이탈 방지 및 고객 충성도를 높인다. | 캠페인 반응이 예상보다 저조할 경우, ROI 감소가 발생할 수 있다. | 고객 이탈의 주요 원인이 특정 제품이나 서비스의 품질 저하라고 가정한다. |
| 1. 고객 데이터 분석: 이탈 이유 및 구매 패턴 파악 | 고객 데이터의 불완전성으로 인한 분석 오류 가능성 | 이탈 고객의 데이터가 충분히 수집되어 있다고 가정한다. |
| 2. 맞춤형 제안: 이탈 고객에게 맞춤형 혜택 제공 | 고객의 선호도가 변화하여 맞춤형 제안이 효과가 없을 수 있음 | 고객의 이전 구매 이력이 여전히 유의미할 것이라고 가정한다. |
| 3. 재참여 유도: 이메일 및 SMS 캠페인 진행 | 스팸으로 인식되어 고객의 반감을 살 우려 | 고객이 이메일 및 SMS를 수신하는 데 긍정적일 것이라고 가정한다. |
| 4. 피드백 수집: 캠페인 후 고객 피드백 요청 | 고객이 피드백에 응답하지 않을 위험 | 고객이 자신의 의견을 공유할 의향이 있으며, 피드백 채널이 용이하다고 가정한다. |
| 5. 성과 분석: 캠페인 후 이탈률 변화 분석 | 성과 분석이 부정확할 경우, 잘못된 결론 도출 | 이탈률 변화가 캠페인 효과를 제대로 반

---
## 7. Constraint Prompting

In [25]:
system_prompt = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤, "
    "리스크와 가정을 명시해야 한다."
)

constraints = """조건(반드시 준수):
- 추가 예산은 최대 1억 원 이내
- 인력 증원 불가 (기존 인력 내에서 운영)
- 2주 이내 실행 가능한 액션만 제시
- 고객에게 직접 연락하는 액션은 과도한 빈도를 피하고(1회 이내),
  개인정보/컴플라이언스 리스크를 고려할 것
"""

user_prompt = f"""이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘.

{constraints}

출력 형식:
- 결론(한 줄) 먼저
- 그 다음 표로 정리 (컬럼 예: 타깃/액션/채널/우선순위/기간/기대효과/담당)
- 마지막에 리스크/가정 명시
"""

constrained_answer = ask(system_prompt, user_prompt, temperature=0.7)
print('\n--- 조건 기반 최종 답변(답변만) ---')
print(constrained_answer)


--- 조건 기반 최종 답변(답변만) ---
결론: VIP 이탈 방지를 위해 1억 원 이내의 예산으로 고객 맞춤형 리워드 프로그램을 실행한다.

| 타깃      | 액션                               | 채널         | 우선순위 | 기간       | 기대효과                                 | 담당          |
|-----------|------------------------------------|--------------|----------|------------|------------------------------------------|---------------|
| VIP 200명 | 맞춤형 리워드 쿠폰 발송          | 이메일       | 1        | 1주 이내   | 이탈 방지 및 재방문 유도                | 마케팅팀      |
| VIP 200명 | 개인 맞춤형 상품 추천             | 모바일 앱    | 2        | 1주 이내   | 고객의 관심도 증대 및 구매 유도         | MD팀          |
| VIP 200명 | 특별 이벤트 초대장 발송           | 이메일       | 3        | 2주 이내   | 고객의 참여 유도 및 충성도 강화         | 이벤트팀      |
| VIP 200명 | 설문조사 진행 (1회)               | 이메일       | 4        | 2주 이내   | 고객의 의견 수렴 및 서비스 개선에 도움   | CS팀          |

### 리스크
- 개인정보 보호 및 컴플라이언스 리스크: 고객의 개인정보를 보호하며, 연락 빈도를 최소화해야 함.
- 리워드 프로그램의 효과 미비: 고객의 실제 반응이 예상보다 낮을 수 있음.

### 가정
- 고객의 반응이 긍정적일 것이라는 가정 하에 계획이 진행됨.
- 예산 내에서 모든 액션이 실행 가능하다고 